# 12 · Logistic regression

A classification-specific linear model with interpretable standardised coefficients. This notebook uses frozen train and validation tabular caches only. No audio, mel cache, test manifest, or test feature archive is loaded.


In [1]:
from pathlib import Path
import sys,json,time,warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from dronedet.config import load_project_config
from dronedet.features import load_tabular
from dronedet.metrics import classification_metrics,select_f1_threshold,miss_rate_at_false_alarm_rate
cfg=load_project_config(ROOT/'config.yaml'); P=cfg['P']


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Frozen inputs and scaling

The scaler is fit on train rows excluding `silent` and `near_silent` only; `crop_all_zero` alone stays eligible. Q-A chooses thresholds on validation F1, ties toward the higher threshold, while ROC-AUC and low false-alarm operating points are also reported.


In [2]:
train=load_tabular(P,'train'); val=load_tabular(P,'val')
Xtr=np.asarray(train['X'],dtype=np.float64); ytr=np.asarray(train['y'],dtype=np.int8)
Xva=np.asarray(val['X'],dtype=np.float64); yva=np.asarray(val['y'],dtype=np.int8)
names=np.asarray(train['feature_names']).astype(str)
assert Xtr.shape==(7607,84) and Xva.shape==(1510,84)
assert np.array_equal(train['clip_id'],pd.read_csv(P['splits']/'train.csv',keep_default_na=False).clip_id)
assert np.array_equal(val['clip_id'],pd.read_csv(P['splits']/'val.csv',keep_default_na=False).clip_id)
excluded=np.asarray(train['silent'],bool)|np.asarray(train['near_silent'],bool); eligible=~excluded
assert int(eligible.sum())==7557
scaler=StandardScaler().fit(Xtr[eligible]); Xtr_s=scaler.transform(Xtr); Xva_s=scaler.transform(Xva)
print('Train/validation:',len(ytr),len(yva),'| scaler rows:',int(eligible.sum()))


Train/validation: 7607 1510 | scaler rows: 7557


## Baseline and grid

The baseline is L2, `C=1`, `lbfgs`, `max_iter=1000`, and balanced class weights. The prewritten 12-setting grid varies `C` over six values and compares L2 with L1 (`liblinear` for L1). These are deterministic fits, so a non-default replaces the baseline only when validation F1 improves by at least 0.01.


In [3]:
Cs=[1e-3,1e-2,1e-1,1,10,100]
configs=[{'C':C,'penalty':penalty} for C in Cs for penalty in ('l2','l1')]
default={'C':1.0,'penalty':'l2'}
rows=[]; models={}; t0=time.perf_counter()
for i,hp in enumerate(configs):
    solver='lbfgs' if hp['penalty']=='l2' else 'liblinear'
    model=LogisticRegression(C=hp['C'],solver=solver,l1_ratio=1.0 if hp['penalty']=='l1' else None,
       max_iter=1000,tol=1e-4,class_weight='balanced',random_state=int(cfg['seed']))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always',ConvergenceWarning); model.fit(Xtr_s,ytr)
    score=model.decision_function(Xva_s); threshold=select_f1_threshold(yva,score)
    m=classification_metrics(yva,score,threshold=threshold)
    rows.append({'config_id':i,**hp,'solver':solver,'threshold':float(threshold),'val_f1':m['f1'],
      'val_roc_auc':m['roc_auc'],'false_alarm_rate':m['false_alarm_rate'],'miss_rate':m['miss_rate'],
      'convergence_warning':any(issubclass(w.category,ConvergenceWarning) for w in caught),
      'n_nonzero_coef':int(np.count_nonzero(model.coef_))})
    models[i]=model
results=pd.DataFrame(rows); default_id=next(r['config_id'] for r in rows if r['C']==1 and r['penalty']=='l2')
default_score=float(results.loc[results.config_id==default_id,'val_f1'].iloc[0])
results['delta_vs_default']=results.val_f1-default_score
results['passes_replace_rule']=(results.config_id!=default_id)&(results.delta_vs_default>=.01)
rank=results.sort_values(['val_f1','config_id'],ascending=[False,True]); candidate=rank.iloc[0]
chosen_id=int(candidate.config_id) if (int(candidate.config_id)==default_id or bool(candidate.passes_replace_rule)) else default_id
chosen=results.loc[results.config_id==chosen_id].iloc[0]; model=models[chosen_id]
print('Grid fits:',len(results),'| elapsed:',round(time.perf_counter()-t0,2),'s')
print('Top:',candidate[['C','penalty','val_f1']].to_dict(),'| frozen:',chosen[['C','penalty','val_f1']].to_dict())


Grid fits: 12 | elapsed: 84.58 s
Top: {'C': 0.01, 'penalty': 'l2', 'val_f1': 0.7425149700598802} | frozen: {'C': 0.01, 'penalty': 'l2', 'val_f1': 0.7425149700598802}


## Validation metrics and diagnostics

The F1 threshold is selected on validation. Probability calibration is descriptive; it does not select the model or threshold.


In [4]:
score=model.decision_function(Xva_s); threshold=select_f1_threshold(yva,score)
metrics=classification_metrics(yva,score,threshold=threshold)
miss1,thr1=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.01)
miss5,thr5=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.05)
prob=model.predict_proba(Xva_s)[:,1]
from sklearn.calibration import calibration_curve
frac_pos,mean_pred=calibration_curve(yva,prob,n_bins=10,strategy='quantile')
FIG=ROOT/'results'/'figures'; TABLES=ROOT/'results'/'tables'; MODELS=ROOT/'results'/'models'; CARDS=ROOT/'results'/'cards'
for d in (FIG,TABLES,MODELS,CARDS): d.mkdir(parents=True,exist_ok=True)
results.to_csv(TABLES/'12_logistic_regression_grid.csv',index=False)
coef=pd.DataFrame({'feature':names,'coefficient':model.coef_[0]}); coef['abs_coefficient']=coef.coefficient.abs()
coef.sort_values('abs_coefficient',ascending=False).to_csv(TABLES/'12_logistic_regression_coefficients.csv',index=False)
fig,ax=plt.subplots(figsize=(7,5)); ax.plot(mean_pred,frac_pos,'o-'); ax.plot([0,1],[0,1],':',color='gray'); ax.set(xlabel='Mean predicted validation probability',ylabel='Observed positive fraction',title='Validation calibration (descriptive)'); fig.tight_layout(); fig.savefig(FIG/'12_logistic_regression_calibration.png',dpi=160); plt.close(fig)
fig,ax=plt.subplots(figsize=(7,5)); ax.plot(results.C.astype(str)+' '+results.penalty,results.val_f1,'o-'); ax.set(xlabel='C and penalty',ylabel='Validation F1',title='Logistic regression grid'); ax.tick_params(axis='x',rotation=60); fig.tight_layout(); fig.savefig(FIG/'12_logistic_regression_grid.png',dpi=160); plt.close(fig)


## Freeze

Freeze the validation-selected logistic model, scaler, feature order, threshold and front-end hash. The test split is not opened.


In [5]:
card={'model':'logistic_regression','representation':'84 tabular features','best_hyperparameters':{'C':float(chosen['C']),'penalty':chosen['penalty'],'solver':chosen['solver']},
 'val_metrics':{**metrics,'threshold':float(threshold),'miss_rate_at_far_1pct':float(miss1),'threshold_at_far_1pct':float(thr1),
 'miss_rate_at_far_5pct':float(miss5),'threshold_at_far_5pct':float(thr5)},'seeds':[int(cfg['seed'])],
 'train_time_s':float(time.perf_counter()-t0),'convergence_warning':bool(chosen.convergence_warning),
 'strengths':['classification-specific loss','standardised coefficients and sparse L1 option'],
 'weaknesses':['balanced class weights shift probability interpretation','linear decision boundary'],
 'decision_log_ids':['D044','D045','D047']}
front_hash=json.loads((P['spec']/'frontend_hash.json').read_text(encoding='utf-8'))['params_hash']
joblib.dump({'model':model,'scaler':scaler,'feature_names':names.tolist(),'threshold':float(threshold),
 'threshold_rule':'validation_f1_max_ties_higher','scaler_fit_exclusion':'silent OR near_silent; crop_all_zero alone eligible',
 'front_end_hash':front_hash,'hyperparameters':card['best_hyperparameters']},MODELS/'12_logistic_regression.joblib')
(CARDS/'12_logistic_regression.json').write_text(json.dumps(card,indent=2,default=float),encoding='utf-8')
print('Validation metrics:',metrics,'| FAR miss:',miss1,miss5)


Validation metrics: {'accuracy': 0.9430463576158941, 'precision': 0.8322147651006712, 'recall': 0.6702702702702703, 'f1': 0.7425149700598802, 'roc_auc': 0.9499439061703213, 'false_alarm_rate': 0.018867924528301886, 'miss_rate': 0.32972972972972975, 'confusion_matrix': [[1300, 25], [61, 124]]} | FAR miss: 0.508108108108108 0.2648648648648648
